# Exp8 DDMOEA-GAN (2024)

Runs the uploaded DDMOEA-GAN baseline method: WGAN-GP, RBFN ensemble, critic fitness and NSGA-II with population size 100 for 100 generations.

Offline dataset settings are loaded from `experiments/config.yaml`, using `max(11 * n_var - 1, 100)` points and the Exp1 training seed. Each method initializes its population with the same number of leading offline points as its configured population size.


### Package


In [ ]:
import os
os.environ.pop("HV_TEST_PLAIN_PLOT", None)

import importlib
import importlib.util
import subprocess
import sys
import types

sys.dont_write_bytecode = True

if "imp" not in sys.modules and importlib.util.find_spec("imp") is None:
    # pyDOE2 1.3.0 imports the removed module but does not use it for LHS.
    sys.modules["imp"] = types.ModuleType("imp")

DEPENDENCIES = {
    "pymoo": "pymoo==0.6.1.6",
    "torch": "torch",
    "yaml": "pyyaml",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "plotly": "plotly"
}

for import_name, pip_name in DEPENDENCIES.items():
    try:
        importlib.import_module(import_name)
        print(f"{import_name} is available.")
    except ImportError:
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", pip_name])


try:
    from google.colab import drive
except ModuleNotFoundError:
    drive = None

if drive is not None:
    drive.mount('/content/drive')

from pathlib import Path

code_path = Path("/content/drive/MyDrive/2026 Indicator_misleading/src")
for repo_root in (Path.cwd().resolve(), Path.cwd().resolve().parent, code_path.parent, code_path):
    if (repo_root / "baseline" / "batch_experiments.py").exists():
        repo_root_string = str(repo_root)
        while repo_root_string in sys.path:
            sys.path.remove(repo_root_string)
        sys.path.insert(0, repo_root_string)
        print(f"Using repository root: {repo_root}")
        break
else:
    raise FileNotFoundError("Could not locate baseline/batch_experiments.py")

importlib.invalidate_caches()
for module_name in list(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        sys.modules.pop(module_name, None)
sys.modules.pop("baseline.batch_experiments", None)
sys.modules.pop("baseline", None)
import baseline.batch_experiments as batch_experiments

print(f"Loaded baseline runner: {batch_experiments.__file__}")
config_path = code_path.parent / "experiments" / "config.yaml"
if not config_path.exists():
    config_path = Path(getattr(batch_experiments, "DEFAULT_CONFIG_PATH"))
print(f"Using experiment config: {config_path}")
run_ddmoea_gan_suite = batch_experiments.run_ddmoea_gan_suite
print_gap_improvement_table = batch_experiments.print_gap_improvement_table



### Run


In [ ]:
all_results = run_ddmoea_gan_suite(config_path=config_path)


### Gap improvement table


In [ ]:
# Per-seed bluebear-style results are printed by the suite.
# Aggregate gap-improvement table printing is skipped for single-seed runs.
gap_improvement_table = None
